# TCA-1 — Translation causal ablation

Diagnostic-only A0/A1 run over `DEV_CROSS_60`. A0 is canonical OPUS; A1 substitutes the frozen reference English only for the exact 17 frozen FAIL semantic units. GT is not opened until both prediction files are finalized and hashed.


In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path
from zipfile import ZIP_DEFLATED, ZipFile, ZipInfo

REPO_URL = os.environ.get('AIC_REPO_URL', 'https://github.com/Irthn1311/AIC2026_TeamPTK_SGU.git')
REPO_REF = os.environ.get('AIC_REPO_REF', 'TRIAGEEG')
REPO_DIR = Path(os.environ.get('AIC_REPO_DIR', '/kaggle/working/AIC2026_TeamPTK_SGU'))
DATA_INPUT = Path(os.environ.get('AIC_DATA_ROOT', '/kaggle/input/datasets/nadkli/dataset-aic'))
TEAM_EVAL_INPUT = Path(os.environ.get('AIC_TEAM_EVAL_DEV_ROOT', '/kaggle/input/datasets/irthn1311/aic2026_team_eval_dev_v1'))
STAGE1_INPUT = Path(os.environ.get('AIC_STAGE1_ROOT', '/kaggle/input/datasets/irthn1311/triage-eg-stage1b-input-bundle'))
STAGE1B_INPUT = Path(os.environ.get('AIC_STAGE1B_ROOT', '/kaggle/input/datasets/irthn1311/triage-eg-stage1b-encoder-compatibility-reports'))
STAGE1E_INPUT = Path(os.environ.get('AIC_STAGE1E_ROOT', '/kaggle/input/datasets/irthn1311/triage-eg-stage1e-language-path-freeze'))
CLIP_INPUT = Path(os.environ.get('AIC_CLIP_ROOT', '/kaggle/input/datasets/irthn1311/aic2026-openai-clip-vit-b32'))
OPUS_INPUT = Path(os.environ.get('AIC_OPUS_ROOT', '/kaggle/input/datasets/irthn1311/aic2026-opus-mt-vi-en'))
FREEZE_INPUT = Path(os.environ.get('AIC_TCA1_FREEZE_ROOT', '/kaggle/input/datasets/irthn1311/triage-eg-tca1-preparation-freeze-2026-08-16'))
OUTPUT_ROOT = Path('/kaggle/working/artifacts/tca1_translation_causal_v01')
ZIP_PATH = Path('/kaggle/working/triage_eg_tca1_translation_causal_v01_bundle.zip')
WORK_ROOT = Path('/kaggle/working/triage_eg_tca1_work')
for target in (OUTPUT_ROOT, WORK_ROOT):
    if target.exists():
        if Path('/kaggle/working') not in target.parents:
            raise RuntimeError(f'Refusing cleanup outside /kaggle/working: {target}')
        shutil.rmtree(target)
ZIP_PATH.unlink(missing_ok=True)
print({'required_inputs': {'raw_dataset': str(DATA_INPUT), 'team_eval_dev_bundle': str(TEAM_EVAL_INPUT), 'stage1_exact_index': str(STAGE1_INPUT), 'stage1b_verified_contract': str(STAGE1B_INPUT), 'stage1e_language_contract': str(STAGE1E_INPUT), 'openai_clip_offline_asset': str(CLIP_INPUT), 'opus_mt_vi_en_offline_asset': str(OPUS_INPUT), 'tca1_preparation_freeze': str(FREEZE_INPUT)}, 'internet_required': 'ONLY_FOR_GIT_CLONE_IF_REPO_NOT_PRESENT', 'model_download_required': False, 'output_zip': str(ZIP_PATH)})


In [ ]:
SEARCH_ROOT = Path('/kaggle/input')
MAX_DEPTH, MAX_DIRECTORIES = 7, 10000
def bounded_dirs(root):
    queue, visited = [(Path(root), 0)], 0
    while queue:
        current, depth = queue.pop(0)
        if not current.is_dir():
            continue
        visited += 1
        if visited > MAX_DIRECTORIES:
            raise RuntimeError('Kaggle input discovery exceeded bound')
        yield current
        if depth < MAX_DEPTH:
            queue.extend((child, depth + 1) for child in sorted(current.iterdir()) if child.is_dir() and not child.is_symlink())
def resolve_file(hint, filename, optional=False):
    hint = Path(hint)
    if hint.is_file() and hint.name == filename:
        return hint.resolve()
    roots, matches = ([hint] if hint.exists() else [SEARCH_ROOT]), []
    for root in roots:
        matches.extend(directory / filename for directory in bounded_dirs(root) if (directory / filename).is_file())
        if matches:
            break
    matches = sorted(set(path.resolve() for path in matches))
    if not matches and optional:
        return None
    if len(matches) != 1:
        raise RuntimeError(f'Expected exactly one {filename}; found {matches}')
    return matches[0]
def resolve_root(hint, marker, optional=False):
    hint, marker = Path(hint), Path(marker)
    roots, matches = ([hint] if hint.exists() else [SEARCH_ROOT]), []
    for root in roots:
        matches.extend(directory for directory in bounded_dirs(root) if (directory / marker).is_file())
        if matches:
            break
    matches = sorted(set(path.resolve() for path in matches))
    if not matches and optional:
        return None
    if len(matches) != 1:
        raise RuntimeError(f'Expected exactly one root with {marker}; found {matches}')
    return matches[0]
def resolve_dataset(hint):
    hint, marker = Path(hint), Path('map-keyframes-aic25-b1/map-keyframes')
    roots, matches = ([hint] if hint.exists() else [SEARCH_ROOT]), []
    for root in roots:
        matches.extend(directory for directory in bounded_dirs(root) if (directory / marker).is_dir() and any(directory.glob('Videos_*/video')))
        if matches:
            break
    matches = sorted(set(path.resolve() for path in matches))
    if len(matches) != 1:
        raise RuntimeError(f'Expected exactly one raw dataset root; found {matches}')
    return matches[0]
DATASET_ROOT = resolve_dataset(DATA_INPUT)
TEAM_EVAL_ZIP_MOUNT = resolve_file(TEAM_EVAL_INPUT, 'aic2026_team_eval_dev_v1.zip', optional=True)
TEAM_EVAL_ROOT_MOUNT = None if TEAM_EVAL_ZIP_MOUNT else resolve_root(TEAM_EVAL_INPUT, 'benchmarks/dev_cross_60/queries.jsonl')
FREEZE_ZIP = resolve_file(FREEZE_INPUT, 'TCA1_PREPARATION_FREEZE_2026-08-16.zip', optional=True)
FREEZE_SOURCE = FREEZE_ZIP or resolve_root(FREEZE_INPUT, 'translation_blind_review_summary.json')
print({'resolved_raw': str(DATASET_ROOT), 'resolved_team_eval_zip': str(TEAM_EVAL_ZIP_MOUNT) if TEAM_EVAL_ZIP_MOUNT else None, 'resolved_team_eval_root': str(TEAM_EVAL_ROOT_MOUNT) if TEAM_EVAL_ROOT_MOUNT else None, 'resolved_tca1_freeze': str(FREEZE_SOURCE)})


In [ ]:
if not (REPO_DIR / '.git').is_dir():
    if REPO_DIR.exists():
        raise RuntimeError(f'Incomplete repository directory: {REPO_DIR}')
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, str(REPO_DIR)], check=True, env={**os.environ, 'GIT_LFS_SKIP_SMUDGE': '1'})
if not (REPO_DIR / 'src/triage_eg/diagnostics/tca1_translation_causal/runner.py').is_file():
    raise RuntimeError('TRIAGEEG ref does not contain TCA-1; publish reviewed source changes before Kaggle execution')
sys.path.insert(0, str(REPO_DIR / 'src'))
HEAD = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, capture_output=True, text=True, check=True).stdout.strip()
BRANCH = subprocess.run(['git', 'branch', '--show-current'], cwd=REPO_DIR, capture_output=True, text=True, check=True).stdout.strip()
GIT_STATUS = subprocess.run(['git', 'status', '--short'], cwd=REPO_DIR, capture_output=True, text=True, check=True).stdout.strip()
if BRANCH != REPO_REF:
    raise RuntimeError(f'Expected branch {REPO_REF}, resolved {BRANCH}')
print({'branch': BRANCH, 'HEAD': HEAD, 'git_status': GIT_STATUS or 'CLEAN'})


In [ ]:
import yaml
from triage_eg.diagnostics.tca1_translation_causal import TCA1Settings, materialize_frozen_review
from triage_eg.retrieval.stage1b.inputs import resolve_stage1_root
from triage_eg.retrieval.stage1d.inputs import resolve_input_root
MATERIALIZED = {name: WORK_ROOT / name for name in ('stage1', 'stage1b', 'stage1e', 'clip', 'opus')}
def optional_search_root(hint):
    return None if Path(hint).exists() else SEARCH_ROOT
STAGE1_ROOT = resolve_stage1_root(STAGE1_INPUT, search_root=optional_search_root(STAGE1_INPUT), materialize_root=MATERIALIZED['stage1'])
STAGE1B_ROOT, _ = resolve_input_root(STAGE1B_INPUT, required=('stage1b_summary.json', 'encoder/selected_encoder_contract.json', 'encoder/runtime_adapter_manifest.json'), materialize_root=MATERIALIZED['stage1b'], search_root=optional_search_root(STAGE1B_INPUT), archive_keyword='stage1b')
STAGE1E_ROOT, _ = resolve_input_root(STAGE1E_INPUT, required=('stage1e_summary.json', 'language_path_contract.json'), materialize_root=MATERIALIZED['stage1e'], search_root=optional_search_root(STAGE1E_INPUT), archive_keyword='stage1e')
CLIP_ROOT, _ = resolve_input_root(CLIP_INPUT, required=('checkpoint/ViT-B-32.pt', 'manifests/asset_manifest.json'), materialize_root=MATERIALIZED['clip'], search_root=optional_search_root(CLIP_INPUT), archive_keyword='clip')
OPUS_ROOT, _ = resolve_input_root(OPUS_INPUT, required=('model/config.json', 'manifests/asset_manifest.json'), materialize_root=MATERIALIZED['opus'], search_root=optional_search_root(OPUS_INPUT), archive_keyword='opus')
SETTINGS = TCA1Settings()
EXPERIMENT_CONFIG = yaml.safe_load((REPO_DIR / 'configs/experiments/triage_tca1_translation_causal_v01.yaml').read_text(encoding='utf-8'))
if EXPERIMENT_CONFIG.get('experiment') != 'TRIAGE_TCA1_TRANSLATION_CAUSAL' or EXPERIMENT_CONFIG.get('benchmark') != SETTINGS.benchmark or EXPERIMENT_CONFIG.get('selected_variant') != SETTINGS.selected_variant or EXPERIMENT_CONFIG.get('production_policy_changed') is not False:
    raise RuntimeError('TCA1 YAML/dataclass contract mismatch')
FROZEN = materialize_frozen_review(FREEZE_SOURCE, OUTPUT_ROOT)
print({'stage1': str(STAGE1_ROOT), 'stage1b': str(STAGE1B_ROOT), 'stage1e': str(STAGE1E_ROOT), 'clip': str(CLIP_ROOT), 'opus': str(OPUS_ROOT), 'freeze_validation': FROZEN.validation})


In [ ]:
from aic2026_eval.io import write_jsonl
from triage_eg.e2eg1 import SafeCoveragePipeline
from triage_eg.retrieval.stage2 import OperationalRetrievalRuntime, config_from_yaml
from triage_eg.diagnostics.tca1_translation_causal import TCA1RuntimeProxy
QUERY_ONLY_ROOT = WORK_ROOT / 'inference_only/dev_cross_60'
QUERY_ONLY_ROOT.mkdir(parents=True, exist_ok=True)
if TEAM_EVAL_ZIP_MOUNT:
    with ZipFile(TEAM_EVAL_ZIP_MOUNT) as archive:
        query_rows = [json.loads(line) for line in archive.read('benchmarks/dev_cross_60/queries.jsonl').decode('utf-8').splitlines() if line]
else:
    query_rows = [json.loads(line) for line in (TEAM_EVAL_ROOT_MOUNT / 'benchmarks/dev_cross_60/queries.jsonl').read_text(encoding='utf-8').splitlines() if line]
write_jsonl(QUERY_ONLY_ROOT / 'queries.jsonl', query_rows)
assert {path.name for path in QUERY_ONLY_ROOT.iterdir()} == {'queries.jsonl'}
def make_pipeline(arm):
    runtime_root = WORK_ROOT / f'runtime_{arm.casefold()}'
    config = config_from_yaml(REPO_DIR / 'configs/retrieval/stage2_operational_runtime_gpu.yaml', stage1_root=STAGE1_ROOT, stage1b_root=STAGE1B_ROOT, stage1e_root=STAGE1E_ROOT, clip_asset_root=CLIP_ROOT, translator_asset_root=OPUS_ROOT, output_root=runtime_root, stage1d_config=REPO_DIR / 'configs/retrieval/stage1d_translation_ablation.yaml', build_git_commit=HEAD)
    delegate = OperationalRetrievalRuntime(config).load()
    proxy = TCA1RuntimeProxy(delegate, FROZEN, arm)
    return SafeCoveragePipeline(proxy, DATASET_ROOT), proxy
A0_PIPELINE, A0_PROXY = make_pipeline('A0')
A1_PIPELINE, A1_PROXY = make_pipeline('A1')
print({'GT_AVAILABLE_TO_PREDICTION': False, 'arm_runtime_identity_distinct': A0_PROXY.delegate is not A1_PROXY.delegate, 'arm_pipeline_identity_distinct': A0_PIPELINE is not A1_PIPELINE})


In [ ]:
from triage_eg.diagnostics.tca1_translation_causal import run_pre_gt_arm, validate_pre_gt_integrity
A0_RUN, A0_SNAPSHOT = run_pre_gt_arm(A0_PIPELINE, QUERY_ONLY_ROOT, OUTPUT_ROOT, WORK_ROOT / 'prediction_temp', 'A0')
A1_RUN, A1_SNAPSHOT = run_pre_gt_arm(A1_PIPELINE, QUERY_ONLY_ROOT, OUTPUT_ROOT, WORK_ROOT / 'prediction_temp', 'A1')
A0_MANIFEST, A1_MANIFEST = A0_PROXY.runtime_manifest(), A1_PROXY.runtime_manifest()
INTEGRITY = validate_pre_gt_integrity(A0_RUN, A1_RUN, A0_SNAPSHOT, A1_SNAPSHOT, FROZEN, A0_MANIFEST, A1_MANIFEST, SETTINGS, A0_PIPELINE.settings.as_dict(), A1_PIPELINE.settings.as_dict())
print({'prediction_hashes': {'A0': A0_RUN['sha256'], 'A1': A1_RUN['sha256']}, 'intervention_integrity': INTEGRITY, 'GT_OPENED': False})


In [ ]:
from triage_eg.e2eg1 import extract_development_bundle
TEAM_EVAL_REPACKED = WORK_ROOT / 'aic2026_team_eval_dev_v1_repacked.zip'
if TEAM_EVAL_ZIP_MOUNT:
    TEAM_EVAL_ZIP = TEAM_EVAL_ZIP_MOUNT
else:
    required = [path.relative_to(TEAM_EVAL_ROOT_MOUNT).as_posix() for path in TEAM_EVAL_ROOT_MOUNT.rglob('*') if path.is_file() and 'sealed' not in path.as_posix().casefold()]
    with ZipFile(TEAM_EVAL_REPACKED, 'w', ZIP_DEFLATED) as archive:
        for member in sorted(required):
            info = ZipInfo(member, date_time=(1980, 1, 1, 0, 0, 0)); info.compress_type = ZIP_DEFLATED; info.external_attr = 0o644 << 16
            archive.writestr(info, (TEAM_EVAL_ROOT_MOUNT / member).read_bytes())
    TEAM_EVAL_ZIP = TEAM_EVAL_REPACKED
TEAM_EVAL_ROOT = extract_development_bundle(TEAM_EVAL_ZIP, WORK_ROOT / 'team_eval_extracted')
CROSS_ROOT = TEAM_EVAL_ROOT / 'benchmarks/dev_cross_60'
print({'GT_OPENED_AFTER_BOTH_HASHES': True, 'cross_root': str(CROSS_ROOT)})


In [ ]:
from triage_eg.diagnostics.tca1_translation_causal import create_bundle, evaluate_post_gt, formal_report, write_readme, write_run_manifests
EVALUATION = evaluate_post_gt(a0_pipeline=A0_PIPELINE, a1_pipeline=A1_PIPELINE, a0_run=A0_RUN, a1_run=A1_RUN, a0_snapshot=A0_SNAPSHOT, a1_snapshot=A1_SNAPSHOT, benchmark_root=CROSS_ROOT, frozen=FROZEN, integrity=INTEGRITY, output_root=OUTPUT_ROOT, temporary_root=WORK_ROOT / 'post_gt')
write_run_manifests(OUTPUT_ROOT, settings=SETTINGS, frozen=FROZEN, integrity=INTEGRITY, a0_run=A0_RUN, a1_run=A1_RUN, a0_runtime_manifest=A0_MANIFEST, a1_runtime_manifest=A1_MANIFEST, branch=BRANCH, git_commit=HEAD, dataset_root=DATASET_ROOT, team_eval_bundle=TEAM_EVAL_ZIP, experiment_config=EXPERIMENT_CONFIG)
write_readme(OUTPUT_ROOT)
BUNDLE = create_bundle(OUTPUT_ROOT, ZIP_PATH)
print(formal_report(head=HEAD, integrity=INTEGRITY, evaluation=EVALUATION, bundle=BUNDLE))
print('INPUTS_USED=', {'raw_dataset': str(DATASET_ROOT), 'team_eval_dev_bundle': str(TEAM_EVAL_ZIP), 'stage1': str(STAGE1_ROOT), 'stage1b': str(STAGE1B_ROOT), 'stage1e': str(STAGE1E_ROOT), 'clip': str(CLIP_ROOT), 'opus': str(OPUS_ROOT), 'tca1_freeze': str(FREEZE_SOURCE)})
A0_PIPELINE.close(); A1_PIPELINE.close()
